# 03 — Healthcare Suitability Model

Builds the healthcare suitability surface from fuzzy-standardised criteria, AHP weights, and constraint masking.

In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
import rasterio
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from utils import load_config, setup_logger
from fuzzy_standardisation import standardise_all_criteria
from suitability_model import (weighted_linear_combination,
                                classify_suitability, build_constraint_mask)
from ahp import build_healthcare_ahp

logger = setup_logger("03_healthcare")
hc_cfg  = load_config(ROOT / "config/healthcare_config.yml")
INTERIM  = ROOT / "data/interim"
PROC     = ROOT / "data/processed"
OUT_R    = ROOT / "outputs/rasters"
OUT_M    = ROOT / "outputs/maps"
PROC.mkdir(exist_ok=True)

print("Healthcare Suitability Model")
print(f"Criteria : {list(hc_cfg['criteria'].keys())}")
print(f"CR       : {hc_cfg['ahp_matrix']['consistency_ratio']}")


Healthcare Suitability Model
Criteria : ['distance_to_health_facility', 'population_density', 'distance_to_roads', 'distance_to_settlements', 'slope', 'land_cover_suitability']
CR       : 0.014


## Step 1 — Verify AHP Weights

In [2]:
# Recompute AHP from scratch to verify config weights
ahp = build_healthcare_ahp()
print(ahp.consistency_report())
print()
print("Weights from config:")
for name, cfg in hc_cfg["criteria"].items():
    print(f"  {name:<40} {cfg['weight']:.4f}  ({cfg['weight']*100:.1f}%)")


AHP CONSISTENCY REPORT
  n criteria   : 6
  λ_max        : 6.0405
  CI           : 0.0081
  RI           : 1.24
  CR           : 0.0065
  Acceptable   : YES ✓

  DERIVED WEIGHTS:
    distance_to_health_facility              0.3290  (32.9%)
    population_density                       0.2437  (24.4%)
    distance_to_roads                        0.1813  (18.1%)
    distance_to_settlements                  0.1063  (10.6%)
    slope                                    0.0836  (8.4%)
    land_cover_suitability                   0.0561  (5.6%)

Weights from config:
  distance_to_health_facility              0.2896  (29.0%)
  population_density                       0.2433  (24.3%)
  distance_to_roads                        0.1707  (17.1%)
  distance_to_settlements                  0.1125  (11.2%)
  slope                                    0.0919  (9.2%)
  land_cover_suitability                   0.0919  (9.2%)


## Step 2 — Fuzzy Standardise All Criteria

In [ ]:
# Map config criterion names to interim raster filenames
RASTER_MAP = {
    "distance_to_health_facility": INTERIM / "dist_health_facility_aligned.tif",
    "population_density":          INTERIM / "population_30m_aligned.tif",
    "distance_to_roads":           INTERIM / "dist_roads_aligned.tif",
    "distance_to_settlements":     INTERIM / "dist_settlements_aligned.tif",
    "slope":                       INTERIM / "slope_degrees_aligned.tif",
    "land_cover_suitability":      INTERIM / "land_cover_30m_aligned.tif",
}

std_rasters = {}
for name, cfg in hc_cfg["criteria"].items():
    src = RASTER_MAP.get(name)
    if src and src.exists():
        from fuzzy_standardisation import standardise_raster
        dst = PROC / f"hc_{name}_std.tif"
        standardise_raster(src, dst, cfg["fuzzy_type"], cfg["fuzzy_params"])
        std_rasters[name] = dst
    else:
        logger.warning(f"Input raster not found for: {name} (expected: {src})")

print(f"\nStandardised {len(std_rasters)} / {len(hc_cfg['criteria'])} criteria")


## Step 3 — Build Constraint Mask

In [ ]:
slope_raster = INTERIM / "slope_degrees_aligned.tif"
lc_raster    = INTERIM / "land_cover_30m_aligned.tif"

constraint_inputs = {}
if slope_raster.exists():
    constraint_inputs["steep_slope"] = (slope_raster, "gt", 20)  # exclude >20 degrees
if lc_raster.exists():
    constraint_inputs["water_bodies"] = (lc_raster, "eq", 70)    # exclude water

ref = INTERIM / "dem_30m.tif"
mask_path = OUT_R / "healthcare_constraint_mask.tif"

if ref.exists() and constraint_inputs:
    build_constraint_mask(constraint_inputs, ref, mask_path)
    print(f"Constraint mask saved: {mask_path}")
else:
    print("Skipping constraint mask — reference raster not yet available.")
    mask_path = None


## Step 4 — Weighted Linear Combination

In [ ]:
weights = {name: cfg["weight"] for name, cfg in hc_cfg["criteria"].items()}
suit_path = OUT_R / "healthcare_suitability.tif"

if std_rasters:
    weighted_linear_combination(
        criteria_raster_paths=std_rasters,
        weights=weights,
        output_path=suit_path,
        constraint_mask_path=mask_path,
    )
    print(f"Suitability raster saved: {suit_path}")
else:
    print("No standardised rasters available yet — run preprocessing first.")


## Step 5 — Classify and Map

In [ ]:
classified_path = OUT_R / "healthcare_classified.tif"

if suit_path.exists():
    classify_suitability(suit_path, classified_path)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Continuous suitability
    with rasterio.open(suit_path) as src:
        arr = src.read(1)
        arr_masked = np.where(arr == src.nodata, np.nan, arr)
    im1 = axes[0].imshow(arr_masked, cmap="RdYlGn", vmin=0, vmax=1)
    axes[0].set_title("Healthcare Suitability (continuous)", fontsize=11)
    axes[0].axis("off")
    plt.colorbar(im1, ax=axes[0], fraction=0.03, label="Suitability score")

    # Classified
    with rasterio.open(classified_path) as src:
        cls = src.read(1)
    cmap_cls = mcolors.ListedColormap(["#d73027","#fc8d59","#fee090","#91cf60","#1a9850"])
    bounds_cls = [0.5,1.5,2.5,3.5,4.5,5.5]
    norm_cls = mcolors.BoundaryNorm(bounds_cls, cmap_cls.N)
    im2 = axes[1].imshow(cls, cmap=cmap_cls, norm=norm_cls)
    axes[1].set_title("Healthcare Suitability (classified)", fontsize=11)
    axes[1].axis("off")
    cbar2 = plt.colorbar(im2, ax=axes[1], fraction=0.03, ticks=[1,2,3,4,5])
    cbar2.set_ticklabels(["Very Low","Low","Moderate","High","Very High"])

    plt.suptitle("Healthcare Suitability — 19 Rural LGAs, Oyo State", fontsize=13)
    plt.tight_layout()
    fig.savefig(OUT_M / "healthcare_suitability_map.png", dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Map saved: {OUT_M / 'healthcare_suitability_map.png'}")
else:
    print("Run WLC step first.")
